# 23 — 2026 raw observed-record source audit + bounded positive-cell diagnostic (RoadAware V3)

Research only (`production_ready = false`). Consumes the **accepted** notebook-22 runtime (selected `hgb_poisson_fixed::routed_context` + locked strong baseline `segment_mean_k365`) and the **accepted** notebook-18 linkage rules, both verified against manager pins before any FUNCTIONS are executed or bundles deserialized. No training, refit, reselection or tuning.

- One CSV row = one **raw observed record**. The file has no `ACC_CODE`: unique physical-event identity is unverified; only exact full-row duplicates are removed.
- Requested window 2026-01-01..2026-09-30 (October excluded by date rule before any score).
- Only observed **positive** eligible linked segment × date (× 6 h block) cells are scored. Unobserved cells stay **unknown** (never zero) → outcome-conditioned selection bias; not comparable with full-population OOF/seen diagnostics.
- Event coordinates and observed weather at the event are never model features.


In [ ]:
# PARAMETERS: roots come from the runner environment (ROADAWARE_BACKEND_ROOT / ROADAWARE_V3_OUTPUT_ROOT); interactive default = this repo's backend/.
import os
from pathlib import Path


def _default_backend(c):
    for p in (c, *c.parents):
        if (p / 'config' / 'spatial_2026_evaluation_v3.json').is_file():
            return p
        if (p / 'backend' / 'config' / 'spatial_2026_evaluation_v3.json').is_file():
            return p / 'backend'
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_NAME = os.environ.get('ROADAWARE_STAGE_CONFIG') or 'spatial_2026_evaluation_v3.json'  # *_v4.json = national
if Path(CONFIG_NAME).name != CONFIG_NAME or not CONFIG_NAME.startswith('spatial_2026_evaluation_v') or not CONFIG_NAME.endswith('.json'):
    raise ValueError('ROADAWARE_STAGE_CONFIG must be a spatial_2026_evaluation_v*.json file name in backend/config')
CONFIG_PATH = BACKEND / 'config' / CONFIG_NAME
print('backend', BACKEND, '| output root', OUTPUT_ROOT)


In [ ]:
# FUNCTIONS (1/4): errors, hashing, strict JSON, NEW-root output guards, manager pin verification (runtime 22 / source / linkage 18) and the
# hash-verified FUNCTIONS loader. Pure logic; the tests exec these cells. Nothing here fits, downloads or writes outside the output dirs.
import datetime as dt
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import re
import time
import types
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

NL = chr(10)
UNITS = ('daily', '6h')


class EvalError(RuntimeError):
    pass


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def jsonable(o):
    # numpy scalars to Python, non-finite floats to None, so json.dumps(allow_nan=False) stays strict
    if isinstance(o, (dict, types.MappingProxyType)):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in (sorted(o) if isinstance(o, set) else o)]
    if isinstance(o, np.ndarray):
        return [jsonable(v) for v in o.tolist()]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    if o is pd.NA or o is pd.NaT:
        return None
    return o


def strict_json_bytes(obj):
    return (json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False) + NL).encode('utf-8')


def read_strict_json(path):
    def bad(c):
        raise EvalError('non-finite JSON constant ' + c + ' in ' + str(path))
    return json.loads(Path(path).read_text('utf-8'), parse_constant=bad)


def atomic_write_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'xb') as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def csv_bytes(df):
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise EvalError('refusing to write infinite values to CSV')
    return df.to_csv(index=False, lineterminator=NL).encode('utf-8')


def module_version(name):
    if importlib.util.find_spec(name) is None:
        return None
    return str(getattr(importlib.import_module(name), '__version__', 'unknown'))


def require_modules(pins):
    missing = [n for n in pins if importlib.util.find_spec(n) is None]
    if missing:
        raise EvalError('missing Python package(s) ' + ', '.join(missing) + '; ask Codex for ' + ' '.join(pins[n] for n in missing))
    if module_version('sklearn') != '1.6.1':
        raise EvalError('scikit-learn ' + str(module_version('sklearn')) + ' != locked 1.6.1')
    return {n: module_version(n) for n in pins}


def environment():
    return {'python': platform.python_version(), 'platform': platform.platform(), 'machine': platform.machine(), 'cpu_count': os.cpu_count(),
            'dependencies': {m: module_version(m) for m in ('pandas', 'numpy', 'sklearn', 'joblib', 'shapely', 'pyproj', 'matplotlib')}}


# ---------- output guards (NEW root, symlinks/aliases, ancestors in both directions) ----------
def output_paths(out_root, cfg):
    out_root = Path(out_root)
    return {'out_root': out_root, 'report_dir': out_root / cfg['outputs']['report_dir'], 'data_dir': out_root / cfg['outputs']['data_dir']}


def _same(a, b):
    try:
        return os.path.samefile(a, b)
    except OSError:
        return False


def _inside(child, parent):
    c, p = Path(child).resolve(), Path(parent).resolve()
    if c == p or p in c.parents:
        return True
    return p.exists() and any(_same(a, p) for a in (c, *c.parents) if a.exists())


def input_rels(cfg):
    R, L, S = cfg['runtime'], cfg['linkage'], cfg['source']
    return [R['notebook22'], R['config22'], R['serving_record'], R['serving_manifest'], R['serving_data_dir'], L['notebook18'], L['config18'],
            L['province_points'], S['new_csv'], S['ckan'], S['original_csv'], cfg['pins']['runtime'], cfg['pins']['source'], cfg['pins']['linkage']]


def guarded_paths(backend, cfg):
    backend = Path(backend)
    rels = sorted(set(cfg['protected_paths']) | set(input_rels(cfg)) | set(cfg['guarded_roots']))
    return [backend / r for r in rels] + [backend / cfg['notebook'], backend / 'config' / CONFIG_NAME]


def check_outputs(paths, backend, cfg):
    """Call before any mkdir/write: report/data dirs must not overlap a protected path, pinned input, accepted root, source notebook or config
    (either direction, symlinks/aliases resolved), must not overlap each other and must be NEW."""
    hits = []
    for label in ('report_dir', 'data_dir'):
        for p in guarded_paths(backend, cfg):
            if _inside(paths[label], p) or _inside(p, paths[label]):
                hits.append(label + ' ' + str(paths[label]) + ' overlaps protected ' + str(p))
    if _inside(paths['report_dir'], paths['data_dir']) or _inside(paths['data_dir'], paths['report_dir']):
        hits.append('report and data dirs overlap')
    if hits:
        raise EvalError('refusing output root (nothing written): ' + '; '.join(hits[:6]))
    used = [str(paths[k]) for k in ('report_dir', 'data_dir') if os.path.lexists(paths[k])]
    if used:
        raise EvalError('output already exists in ' + ', '.join(used) + '; use a NEW root, e.g. backend/rerun-2026-eval-<id>')
    return True


def check_report_file(rel, cfg):
    if Path(rel).suffix.lower() not in cfg['report_allowed_suffixes'] or str(rel).endswith('.gz') or '/' in str(rel):
        raise EvalError('refusing to write ' + str(rel) + ' into the report dir (flat aggregated csv/md/json/png/svg only)')
    return True


def hash_paths(root, rels):
    root, out = Path(root), {}
    for rel in rels:
        p = root / rel
        if p.is_file():
            out[Path(rel).as_posix()] = sha256_file(p)
        elif p.is_dir():
            for q in sorted(p.rglob('*')):
                if q.is_file():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = sha256_file(q)
        else:
            out[Path(rel).as_posix()] = None
    return out


# ---------- manager pins ----------
def verify_file(path, expected, label):
    """Fail closed: no trusted sha -> refused; missing or differing file -> refused."""
    path = Path(path)
    if not isinstance(expected, str) or not re.fullmatch(r'[0-9a-f]{64}', expected):
        raise EvalError(label + ': no trusted sha256 (fail closed)')
    if not path.is_file() or path.is_symlink():
        raise EvalError(label + ': missing or not a regular file ' + str(path))
    got = sha256_file(path)
    if got != expected:
        raise EvalError(label + ': sha256 ' + got[:12] + ' != trusted ' + expected[:12] + ' (' + str(path) + ')')
    return got


def pin_key(cfg, rel):
    return cfg['pins']['repo_prefix'] + Path(rel).as_posix()


def load_pins(backend, cfg, which):
    p = Path(backend) / cfg['pins'][which]
    pins = read_strict_json(p)
    if not isinstance(pins, dict) or pins.get('kind') != cfg['pins'][which + '_kind'] or not isinstance(pins.get('files'), dict) or not pins['files']:
        raise EvalError(which + ' pins missing or of the wrong kind: ' + str(p))
    return pins, sha256_file(p)


def verify_pinned_files(backend, pins, label):
    """Every manager-pinned file (repo-relative key) must exist and hash-match. Returns verification rows."""
    repo, rows = Path(backend).parent, []
    for key, h in sorted(pins['files'].items()):
        if not isinstance(key, str) or Path(key).is_absolute() or '..' in Path(key).parts:
            raise EvalError(label + ' pin key must be repo-relative: ' + str(key))
        rows.append({'pins': label, 'input': key, 'sha256': verify_file(repo / key, h, label + ' pin ' + key), 'rule': 'manager pin'})
    return rows


def require_pinned(cfg, pins, rels, label):
    miss = [r for r in rels if pin_key(cfg, r) not in pins['files']]
    if miss:
        raise EvalError(label + ': consumed inputs without a manager pin: ' + str(miss))
    return True


def verify_source_pins(backend, cfg, pins):
    S = cfg['source']
    require_pinned(cfg, pins, [S['new_csv'], S['ckan'], S['original_csv']], 'source')
    if pins.get('registered_before_any_2026_model_scores') is not True:
        raise EvalError('source pins were not registered before 2026 scores')
    if pins.get('requested_start') != cfg['window']['start'] or pins.get('requested_end') != cfg['window']['end']:
        raise EvalError('source pins requested window differs from the config window')
    if pins.get('new_snapshot', {}).get('resource_id') != S['resource_id']:
        raise EvalError('source pins resource id differs from the config')
    return verify_pinned_files(backend, pins, 'source')


def verify_runtime_pins(backend, cfg, pins):
    R = cfg['runtime']
    require_pinned(cfg, pins, [R['notebook22'], R['config22'], R['serving_record'], R['serving_manifest']], 'runtime')
    f = pins.get('functions') or {}
    if int(f.get('cells', -1)) != int(R['functions_cells']) or f.get('join') != R['functions_join'] or not isinstance(f.get('sha256'), str):
        raise EvalError('runtime pins FUNCTIONS spec differs from the config (cells/join)')
    acc = pins.get('acceptance') or {}
    if acc.get('research_ready') is not True or acc.get('production_ready') is not False:
        raise EvalError('runtime pins do not record research_ready=true / production_ready=false')
    if pin_key(cfg, R['serving_data_dir']) != pins.get('data_dir'):
        raise EvalError('runtime pins data_dir differs from the config serving_data_dir')
    return verify_pinned_files(backend, pins, 'runtime')


def verify_linkage_pins(backend, cfg, pins):
    L = cfg['linkage']
    require_pinned(cfg, pins, [L['notebook18'], L['config18'], L['province_points']], 'linkage')
    f = pins.get('notebook18_functions') or {}
    if int(f.get('cells', -1)) != int(L['functions_cells']) or f.get('join') != L['functions_join'] or not isinstance(f.get('sha256'), str):
        raise EvalError('linkage pins FUNCTIONS spec differs from the config (cells/join)')
    return verify_pinned_files(backend, pins, 'linkage')


def baseline_files_from_record(backend, cfg, runtime_pins):
    """{unit: (path, trusted sha)} from the PINNED serving record; each sha must also equal the manager pin of that npz."""
    backend, R = Path(backend), cfg['runtime']
    rec = read_strict_json(backend / R['serving_record'])
    out = {}
    for u in UNITS:
        e = rec['baseline_stats'][u]
        rel = (Path(R['serving_data_dir']) / e['file']).as_posix()
        if runtime_pins['files'].get(pin_key(cfg, rel)) != e['sha256']:
            raise EvalError('serving record baseline ' + u + ' sha differs from the manager pin')
        out[u] = (backend / rel, e['sha256'])
    return out, rec


# ---------- hash-verified FUNCTIONS ----------
def function_sources(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    cells = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    return [s for s in cells if s.lstrip().startswith('# FUNCTIONS')]


def load_functions(nb_path, n_cells, join, expected_sha, label):
    """exec ONLY the '# FUNCTIONS' cells into a fresh namespace after the cell count AND the joined sha equal the trusted pin (checked first)."""
    funcs = function_sources(nb_path)
    sha = sha256_bytes(join.join(funcs).encode('utf-8'))
    if len(funcs) != int(n_cells) or not isinstance(expected_sha, str) or sha != expected_sha:
        raise EvalError(label + ' FUNCTIONS sha ' + sha[:12] + ' (' + str(len(funcs)) + ' cells) != trusted ' + str(expected_sha)[:12])
    ns = {'__name__': label.replace('-', '_') + '_functions'}
    for i, s in enumerate(funcs):
        exec(compile(s, label + ':FUNCTIONS[' + str(i) + ']', 'exec'), ns)
    return ns, sha


In [ ]:
# FUNCTIONS (2/4): raw CSV reading (every cell kept as the exact text), CKAN provenance, explicit D/M/Y date and H:MM time parsing, exact full-row
# fingerprints, the notebook-16 coordinate flag replica, province validity, report lag/maturity and the old/new snapshot comparison.
# One CSV row = one RAW OBSERVED RECORD (no ACC_CODE: unique event identity unverified). Unknown values stay explicit UNKNOWN; never zero.
import csv
import io

DATE_RE = re.compile(r'^([0-9]{1,2})/([0-9]{1,2})/([0-9]{4})$')
TIME_RE = re.compile(r'^([0-9]{1,2}):([0-9]{2})(?::([0-9]{2}))?$')
FIELD_SEP = '\x1f'
UNKNOWN = 'UNKNOWN'
COORD_STATUSES = ('missing', 'non_numeric', 'non_finite', 'zero_coordinate', 'swap_candidate_unrepaired', 'outside_broad_bbox',
                  'inside_broad_bbox_unverified')


def read_raw_csv(path, encoding, expected_columns):
    """(header, DataFrame of exact strings, data row numbers 1..n). Ragged rows, duplicate/blank headers or a wrong width fail closed."""
    text = Path(path).read_bytes().decode(encoding)
    rows = list(csv.reader(io.StringIO(text, newline='')))
    if not rows:
        raise EvalError('empty CSV ' + str(path))
    header, body = rows[0], rows[1:]
    if len(header) != int(expected_columns) or len(set(header)) != len(header) or any(h.strip() == '' for h in header):
        raise EvalError('header of ' + str(path) + ' has ' + str(len(header)) + ' columns (expected ' + str(expected_columns) + ', unique, non-blank)')
    bad = [i + 1 for i, r in enumerate(body) if len(r) != len(header)]
    if bad:
        raise EvalError(str(len(bad)) + ' ragged rows in ' + str(path) + ' (first data rows ' + str(bad[:5]) + ')')
    df = pd.DataFrame(body, columns=header, dtype=object)
    df.insert(0, '_source_row', np.arange(1, len(df) + 1, dtype=np.int64))
    return header, df


def row_fingerprints(df, header):
    """Deterministic sha256 of the exact raw field values (header order). Only identical full rows share a fingerprint (no fuzzy/location dedup)."""
    vals = df[list(header)].astype(str).to_numpy(object)
    return pd.Series([sha256_bytes(FIELD_SEP.join(r).encode('utf-8')) for r in vals], index=df.index)


def parse_dmy(s):
    """Explicit D/M/YYYY. Returns (iso date or None, status) per value: ok | missing | bad_format | invalid_calendar (e.g. 30/2, 29/2 non-leap)."""
    iso, st = [], []
    for v in pd.Series(s).astype(str).tolist():
        t = v.strip()
        m = DATE_RE.match(t)
        if t == '':
            iso.append(None), st.append('missing')
        elif m is None:
            iso.append(None), st.append('bad_format')
        else:
            try:
                iso.append(dt.date(int(m.group(3)), int(m.group(2)), int(m.group(1))).isoformat()), st.append('ok')
            except ValueError:
                iso.append(None), st.append('invalid_calendar')
    return pd.Series(iso, index=pd.Series(s).index, dtype=object), pd.Series(st, index=pd.Series(s).index, dtype=object)


def parse_hm(s):
    """Explicit H:MM[:SS] local time. Returns (seconds of day as float, NaN when unknown; status ok | missing | bad_format | out_of_range).
    Precision is preserved (seconds kept when given); 24:00 and above are out_of_range."""
    sec, st = [], []
    for v in pd.Series(s).astype(str).tolist():
        t = v.strip()
        m = TIME_RE.match(t)
        if t == '':
            sec.append(np.nan), st.append('missing')
        elif m is None:
            sec.append(np.nan), st.append('bad_format')
        else:
            h, mi, se = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
            if h < 24 and mi < 60 and se < 60:
                sec.append(float(h * 3600 + mi * 60 + se)), st.append('ok')
            else:
                sec.append(np.nan), st.append('out_of_range')
    return pd.Series(sec, index=pd.Series(s).index, dtype=float), pd.Series(st, index=pd.Series(s).index, dtype=object)


def block_of(seconds, block_hours=6):
    """Accepted notebook-10 rule: block b = [b*6 h, (b+1)*6 h); 06:00:00 belongs to 06-12. Unknown time -> None (never a guessed block)."""
    out = []
    for x in np.asarray(seconds, float):
        if not np.isfinite(x) or not 0 <= x < 86400:
            out.append(None)
        else:
            b = int(x) // (block_hours * 3600)
            out.append(format(b * block_hours, '02d') + '-' + format((b + 1) * block_hours, '02d'))
    return out


def norm_text(v):
    return ' '.join(unicodedata.normalize('NFKC', '' if v is None else str(v)).split())


def ckan_provenance(pkg, resource_id, csv_path, expected):
    """Saved CKAN package_show: the resource must exist and agree with the pinned snapshot (size, last_modified). md5 equality is reported only."""
    res = [r for r in (pkg.get('result') or {}).get('resources', []) if r.get('id') == resource_id]
    if pkg.get('success') is not True or len(res) != 1:
        raise EvalError('saved CKAN package does not contain exactly one resource ' + resource_id)
    r = res[0]
    raw = Path(csv_path).read_bytes()
    out = {'resource_id': resource_id, 'name': r.get('name'), 'format': r.get('format'), 'url': r.get('url'), 'last_modified': r.get('last_modified'),
           'ckan_size': r.get('size'), 'file_bytes': len(raw), 'ckan_hash': r.get('hash'), 'file_md5': hashlib.md5(raw).hexdigest(),
           'package_metadata_modified': (pkg.get('result') or {}).get('metadata_modified')}
    out['ckan_hash_equals_file_md5'] = bool(out['ckan_hash'] == out['file_md5'])
    ok = (r.get('format') == 'CSV' and int(r.get('size') or -1) == len(raw) == int(expected['bytes']) and r.get('last_modified') == expected['last_modified']
          and resource_id in str(r.get('url')))
    if not ok:
        raise EvalError('CKAN resource metadata disagrees with the file/pins: ' + json.dumps(jsonable(out))[:400])
    return out


def audit_coordinates(df, lat_col, lon_col, cc):
    """Replica of the accepted notebook-16 audit_coordinates flags (constants from config). Flags only: nothing swapped, moved or rounded."""
    bbox = cc['thailand_broad_bbox']
    lat_s = df[lat_col].fillna('').astype(str).str.strip()
    lon_s = df[lon_col].fillna('').astype(str).str.strip()
    lat = pd.to_numeric(lat_s.where(lat_s != ''), errors='coerce')
    lon = pd.to_numeric(lon_s.where(lon_s != ''), errors='coerce')
    inb = lambda a, o: (a >= bbox['lat_min']) & (a <= bbox['lat_max']) & (o >= bbox['lon_min']) & (o <= bbox['lon_max'])
    missing = (lat_s == '') | (lon_s == '')
    non_numeric = ~missing & (lat.isna() | lon.isna())
    numeric = ~missing & ~non_numeric
    finite = numeric & np.isfinite(lat.fillna(0.0)) & np.isfinite(lon.fillna(0.0))
    inside = finite & inb(lat, lon)
    zero = finite & ((lat.abs() < 1e-9) | (lon.abs() < 1e-9))
    swapped = finite & ~inside & ~zero & inb(lon, lat)
    status = np.select([missing.to_numpy(), non_numeric.to_numpy(), (numeric & ~finite).to_numpy(), zero.to_numpy(), swapped.to_numpy(),
                        (finite & ~inside).to_numpy()], list(COORD_STATUSES[:-1]), default=COORD_STATUSES[-1])

    def decimals_of(s):
        frac = s.str.split('.', n=1).str[1].fillna('')
        d = frac.str.len().where(s.str.contains('.', regex=False), 0)
        return d.where(~s.str.contains('[eE]', regex=True), -1).astype(int)
    lat_dec, lon_dec = decimals_of(lat_s), decimals_of(lon_s)
    key = (lat.round(cc['round_dp']).astype(str) + ',' + lon.round(cc['round_dp']).astype(str)).where(finite, '')
    n_same = key.map(key[finite].value_counts()).where(finite, 0).fillna(0).astype(int)
    low = inside & (lat_dec.between(0, cc['low_precision_max_decimals']) | lon_dec.between(0, cc['low_precision_max_decimals']))
    return pd.DataFrame({'lat_value': lat, 'lon_value': lon, 'coord_status': status, 'lat_decimals': lat_dec, 'lon_decimals': lon_dec,
                         'low_precision_flag': low, 'lat_equals_lon_flag': finite & (lat == lon),
                         'n_events_same_coord': n_same, 'repeated_coord_flag': n_same >= cc['repeated_min_events']}, index=df.index)


def parse_records(df, header, cfg, province_keys):
    """Per raw row: fingerprint, explicit event/report dates and times, block, year check, province validity, agency/route raw text."""
    C = cfg['source']['columns']
    out = pd.DataFrame({'source_row': df['_source_row'].to_numpy(np.int64)}, index=df.index)
    out['fingerprint'] = row_fingerprints(df, header)
    out['exact_duplicate_after_first'] = out['fingerprint'].duplicated(keep='first').to_numpy(bool)
    out['event_date'], out['event_date_status'] = parse_dmy(df[C['event_date']])
    out['event_time_sec'], out['event_time_status'] = parse_hm(df[C['event_time']])
    out['report_date'], out['report_date_status'] = parse_dmy(df[C['report_date']])
    out['report_time_sec'], out['report_time_status'] = parse_hm(df[C['report_time']])
    out['time_block'] = block_of(out['event_time_sec'].to_numpy(float))
    yr = df[C['year']].astype(str).str.strip()
    out['year_matches_date'] = [d is not None and y == d[:4] for d, y in zip(out['event_date'], yr)]
    out['event_month'] = [d[:7] if d else UNKNOWN for d in out['event_date']]
    prov = df[C['province']].map(norm_text)
    out['province_valid'] = prov.isin(province_keys).to_numpy(bool)
    out['province'] = np.where(out['province_valid'], prov, UNKNOWN)
    ag = df[C['agency']].map(norm_text)
    out['agency'] = np.where(ag == '', UNKNOWN, ag)
    out['raw_agency'] = df[C['agency']].astype(str).to_numpy(object)
    out['raw_route_code'] = df[C['route_code']].astype(str).to_numpy(object)
    out['invalid_date_time'] = ((out['event_date_status'] != 'ok') | (out['event_time_status'] != 'ok') | ~out['year_matches_date']).to_numpy(bool)
    return out


def lag_days(R):
    """Report lag (report date - event date, days) only where both parse; unknown report date -> NaN with explicit status."""
    ev = pd.to_datetime(R['event_date'], format='%Y-%m-%d', errors='coerce')
    rp = pd.to_datetime(R['report_date'], format='%Y-%m-%d', errors='coerce')
    lag = (rp - ev).dt.days.astype(float)
    st = np.where(rp.isna(), 'report_date_unknown', np.where(ev.isna(), 'event_date_unknown', np.where(lag < 0, 'negative_lag_report_before_event', 'ok')))
    return lag, pd.Series(st, index=R.index)


def _q(a, q):
    return float(np.quantile(a, q)) if len(a) else None


def lag_table(R, lag, st, by, snapshot_date, label_days, ref_months):
    """Lag distribution and maturity per group (only parseable non-negative lags). late_risk = mean over the group's records of the share of
    reference-month lags (Jan-Mar events, least right-truncated) exceeding the record's age at the snapshot: a RISK INDICATOR, never a correction.
    The reference is itself right-truncated; it is NOT an estimated fraction of unreported true events, a probability or a completeness estimate.
    Missing lag statistics stay missing (NaN in the frame, null in strict JSON, empty in CSV); never 0."""
    snap = pd.Timestamp(snapshot_date)
    ev = pd.to_datetime(R['event_date'], format='%Y-%m-%d', errors='coerce')
    ok = (st == 'ok').to_numpy()
    mon = ev.dt.month.to_numpy()
    ref = np.sort(lag.to_numpy()[ok & np.isin(mon, ref_months)])
    age = (snap - ev).dt.days.to_numpy(float)
    rows = []
    for g, idx in R.groupby(by).groups.items():
        ix = R.index.get_indexer(idx)
        a = lag.to_numpy()[ix][ok[ix]]
        ag = age[ix][np.isfinite(age[ix])]
        risk = float(np.mean([(len(ref) - np.searchsorted(ref, x, side='right')) / len(ref) for x in ag])) if len(ref) and len(ag) else None
        rows.append({by: g, 'n_raw_records': int(len(ix)), 'n_lag_known': int(len(a)),
                     'n_report_date_unknown': int((st.to_numpy()[ix] == 'report_date_unknown').sum()),
                     'n_negative_lag': int((st.to_numpy()[ix] == 'negative_lag_report_before_event').sum()),
                     'lag_median_days': _q(a, 0.5), 'lag_p90_days': _q(a, 0.9), 'lag_p95_days': _q(a, 0.95), 'lag_p99_days': _q(a, 0.99),
                     'lag_max_days': float(a.max()) if len(a) else None, 'share_lag_gt_label_days': float((a > label_days).mean()) if len(a) else None,
                     'min_age_at_snapshot_days': float(ag.min()) if len(ag) else None, 'late_arrival_risk_indicator': risk,
                     'maturity': 'unverified'})
    return pd.DataFrame(rows)


def monthly_counts(R):
    return R.groupby('event_month').size()


def old_new_comparison(R_old, R_new):
    """Aggregate NET record-count change per event month + exact full-row fingerprint multiset overlap. Without ACC_CODE a changed field and a
    new record cannot be told apart: these are revisions/net changes, never proven unique late-arriving events."""
    months = sorted(set(R_old['event_month']) | set(R_new['event_month']))
    rows = []
    for m in months:
        a = R_old.loc[R_old['event_month'] == m, 'fingerprint'].value_counts()
        b = R_new.loc[R_new['event_month'] == m, 'fingerprint'].value_counts()
        both = int(pd.concat([a, b], axis=1).fillna(0).min(axis=1).sum())
        rows.append({'event_month': m, 'old_raw_records': int(a.sum()), 'new_raw_records': int(b.sum()), 'net_record_count_change': int(b.sum() - a.sum()),
                     'identical_full_rows_in_both': both, 'old_rows_without_identical_new_row': int(a.sum()) - both,
                     'new_rows_without_identical_old_row': int(b.sum()) - both,
                     'interpretation': 'net raw record-count change/revision; not proven unique late-arriving events (no ACC_CODE)'})
    return pd.DataFrame(rows)


def date_audit(R, window):
    ev = R['event_date'].dropna()
    rp = R['report_date'].dropna()
    inw = R['event_date'].map(lambda d: d is not None and window['start'] <= d <= window['end'])
    return {'raw_rows': int(len(R)), 'event_date_status': R['event_date_status'].value_counts().to_dict(),
            'event_time_status': R['event_time_status'].value_counts().to_dict(), 'report_date_status': R['report_date_status'].value_counts().to_dict(),
            'report_time_status': R['report_time_status'].value_counts().to_dict(), 'year_mismatch': int((~R['year_matches_date'].astype(bool)).sum()),
            'event_date_min': ev.min() if len(ev) else None, 'event_date_max': ev.max() if len(ev) else None,
            'report_date_min': rp.min() if len(rp) else None, 'report_date_max': rp.max() if len(rp) else None,
            'rows_in_requested_window': int(inw.sum()), 'rows_after_window': int(R['event_date'].map(lambda d: d is not None and d > window['end']).sum()),
            'rows_before_window': int(R['event_date'].map(lambda d: d is not None and d < window['start']).sum()),
            'exact_full_row_duplicates_after_first': int(R['exact_duplicate_after_first'].sum())}


In [ ]:
# FUNCTIONS (3/4): accepted notebook-18 linkage applied to 2026 raw records (exact agency/route identity, 50 m, 5 m ambiguity margin, no snapping),
# the exclusive ordered eligibility ledger + multi-issue flags, observed POSITIVE cells only (never zero-filled), the accepted notebook-22 runtime on
# identical ordered keys for the selected policy and the strong baseline, and per-group metrics. Event coordinates are NEVER model features.
EXCLUSIVE_REASONS = ('outside_window', 'exact_duplicate_after_first', 'invalid_date_time', 'agency_outside_doh_scope', 'invalid_or_unknown_route',
                     'invalid_coordinates_or_province', 'outside_pilot_envelope', 'linkage_outside_50m', 'linkage_ambiguous', 'linkage_unknown',
                     'no_accepted_segment', 'eligible')
ISSUE_FLAGS = EXCLUSIVE_REASONS[:-1]
WEEKDAYS = ('Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun')


def lineage_like(R, coords):
    """Columns the accepted notebook-18 coordinate_audit/event_identity read (flags written as text exactly like the notebook-16 lineage)."""
    out = pd.DataFrame({'raw_agency': R['raw_agency'], 'raw_route_code': R['raw_route_code'], 'lat_value': coords['lat_value'], 'lon_value': coords['lon_value'],
                        'coord_status': coords['coord_status']}, index=R.index)
    for c in ('low_precision_flag', 'lat_equals_lon_flag', 'repeated_coord_flag'):
        out[c] = coords[c].map(lambda b: 'True' if bool(b) else 'False')
    return out


def build_road_model(N18, backend, cfg18):
    """Accepted roads exactly as notebook 18 builds them (derived files verified against config18 AND the road manifest by verify_inputs)."""
    rows, docs, expect = N18['verify_inputs'](backend, cfg18)
    feats, diag = N18['load_features'](Path(backend) / cfg18['inputs']['road_derived_dir'], cfg18, expect['road_counts'])
    fwd, inv = N18['make_transformers'](cfg18)
    _, parts, pieces = N18['build_roads'](feats, diag, cfg18, fwd)
    _, flags, failed = N18['piece_overlaps'](pieces, cfg18['segmentation']['overlap_min_length_m'])
    pieces = pd.concat([pieces, flags], axis=1)
    return {'parts': parts, 'pieces': pieces, 'fwd': fwd, 'inv': inv, 'route_index': N18['route_index'](parts), 'verify_rows': rows,
            'overlap_failed': int(failed), 'road_counts': expect['road_counts']}


def link_records(N18, cfg18, RM, ev_like):
    ev_like = ev_like.reset_index(drop=True)
    coords = N18['coordinate_audit'](ev_like, cfg18)
    ident = N18['event_identity'](ev_like, cfg18, RM['route_index'])
    EV = pd.concat([ev_like, coords, ident], axis=1)
    link = N18['link_events'](EV, RM['parts'], RM['pieces'], cfg18, RM['fwd'], N18['tolerances'](cfg18))
    return pd.concat([EV, link], axis=1)


def recheck_segment_distance(N18, cfg18, RM, E, eligible, seg_col, tol_m, extra_m):
    """Independent geometric re-check: each eligible record's projected point lies within primary tolerance (+tiny float tol) of the geometry
    of the 2000 m segment it was assigned (segments rebuilt with the accepted build_segments). Returns (max distance, n checked, n failing)."""
    from shapely.geometry import LineString, Point
    c2 = json.loads(json.dumps(cfg18))
    R = int(seg_col.split('_')[-1].rstrip('m'))
    c2['segmentation']['resolutions_m'] = [R]
    segs = N18['build_segments'](RM['pieces'], c2, RM['inv'])[R].set_index('segment_id')
    ix = np.flatnonzero(eligible)
    if not len(ix):
        return {'checked': 0, 'max_distance_m': None, 'failing': 0, 'missing_segment': 0}
    xy = N18['project_xy'](RM['fwd'], np.column_stack([E['lon'].to_numpy(float)[ix], E['lat'].to_numpy(float)[ix]]))
    d, miss = [], 0
    for k, i in enumerate(ix):
        sid = E[seg_col].iat[i]
        if sid not in segs.index:
            miss += 1
            continue
        d.append(LineString(segs.at[sid, 'coords_utm']).distance(Point(xy[k])))
    d = np.asarray(d, float)
    lim = float(cfg18['matching']['primary_tolerance_m']) + float(tol_m) + float(extra_m)
    return {'checked': int(len(d)), 'max_distance_m': float(d.max()) if len(d) else None, 'failing': int((d > lim).sum()), 'missing_segment': int(miss),
            'limit_m': lim}


def issue_flags(R, E, window, cfg, universe):
    """Independent boolean issue flags per raw record (several may hold). Linkage flags come from the accepted notebook-18 statuses."""
    L, doh = cfg['linkage'], cfg['linkage']['doh_source_id']
    st = E[L['primary_status_column']].astype(str).to_numpy()
    seg = E[L['segment_column']].to_numpy(object)
    ed = R['event_date'].to_numpy(object)
    F = pd.DataFrame(index=R.index)
    F['outside_window'] = [d is not None and not (window['start'] <= d <= window['end']) for d in ed]
    F['exact_duplicate_after_first'] = R['exact_duplicate_after_first'].to_numpy(bool)
    F['invalid_date_time'] = R['invalid_date_time'].to_numpy(bool)
    F['agency_outside_doh_scope'] = (E['source_id'].to_numpy(object) != doh)
    F['invalid_or_unknown_route'] = (E['identity_status'].astype(str).to_numpy() != 'verified')
    bad_coord = (E['coord_status'].astype(str).to_numpy() != COORD_STATUSES[-1]) | (E['precision_class'].astype(str).to_numpy() == 'low_precision_unsuitable')
    F['invalid_coordinates_or_province'] = bad_coord | ~R['province_valid'].to_numpy(bool)
    F['outside_pilot_envelope'] = (E['scope'].astype(str).to_numpy() == 'outside_envelope')
    linked_scope = ~F['agency_outside_doh_scope'].to_numpy() & ~F['invalid_or_unknown_route'].to_numpy() & (E['scope'].astype(str).to_numpy() == 'pilot') & ~bad_coord
    F['linkage_outside_50m'] = linked_scope & (st == 'too_far')
    F['linkage_ambiguous'] = linked_scope & np.isin(st, ['ambiguous_multiple', 'ambiguous_unsafe_geometry'])
    F['linkage_unknown'] = linked_scope & ~np.isin(st, ['too_far', 'ambiguous_multiple', 'ambiguous_unsafe_geometry', 'matched'])
    assigned = (E['segment_status'].astype(str).to_numpy() == 'assigned') & np.array([s in universe for s in seg], bool)
    F['no_accepted_segment'] = linked_scope & (st == 'matched') & ~assigned
    return F


def exclusive_reason(F):
    """First issue in the preregistered order; 'eligible' when none. Exactly one reason per raw record."""
    out = np.full(len(F), 'eligible', dtype=object)
    done = np.zeros(len(F), bool)
    for r in ISSUE_FLAGS:
        hit = F[r].to_numpy(bool) & ~done
        out[hit] = r
        done |= hit
    return pd.Series(out, index=F.index)


def ledger_tables(R, F, reason):
    L = pd.DataFrame({'event_month': R['event_month'], 'province': R['province'], 'agency': R['agency'], 'reason': reason})
    tabs = {}
    for dim in ('event_month', 'province', 'agency'):
        t = L.groupby([dim, 'reason']).size().unstack(fill_value=0).reindex(columns=list(EXCLUSIVE_REASONS), fill_value=0)
        t.insert(0, 'raw_records', t.sum(axis=1))
        t['excluded'] = t['raw_records'] - t['eligible']
        tabs[dim] = t.reset_index()
    order = pd.DataFrame({'step': range(1, len(EXCLUSIVE_REASONS) + 1), 'reason': EXCLUSIVE_REASONS})
    cnt = reason.value_counts()
    order['records'] = order['reason'].map(cnt).fillna(0).astype(int)
    order['remaining_after_step'] = len(R) - order['records'].where(order['reason'] != 'eligible', 0).cumsum()
    order['any_issue_flag_records'] = [int(F[r].sum()) if r in F else int((F.sum(axis=1) == 0).sum()) for r in order['reason']]
    n_iss = F.sum(axis=1)
    multi = pd.DataFrame({'n_issue_flags': n_iss}).value_counts().sort_index().rename('records').reset_index()
    cross = pd.DataFrame({r: [int((F[r] & (reason == x)).sum()) for x in EXCLUSIVE_REASONS] for r in ISSUE_FLAGS}, index=list(EXCLUSIVE_REASONS))
    cross.index.name = 'exclusive_reason'
    return tabs, order, multi, cross.reset_index()


def observed_cells(R, E, eligible, cfg, unit):
    """Observed POSITIVE eligible linked cells: y = number of eligible raw records per segment x local date (x 6 h block). Absent cells are
    UNKNOWN, never zero. Province/route of a cell = the most frequent value of its records (ties -> sorted first); multi-province cells counted."""
    seg_col = cfg['linkage']['segment_column']
    D = pd.DataFrame({'segment_id': E[seg_col].to_numpy(object)[eligible], 'target_date': R['event_date'].to_numpy(object)[eligible],
                      'time_block': np.asarray(R['time_block'].to_numpy(object))[eligible], 'province': R['province'].to_numpy(object)[eligible],
                      'route': E['route_key_matched'].to_numpy(object)[eligible]})
    keys = ['segment_id', 'target_date'] + (['time_block'] if unit == '6h' else [])
    mode = lambda s: sorted(s.value_counts()[lambda v: v == v.max()].index.astype(str))[0]
    C = D.groupby(keys, sort=True).agg(y=('segment_id', 'size'), province=('province', mode), route=('route', mode),
                                        n_provinces=('province', 'nunique')).reset_index()
    if unit == 'daily':
        C.insert(2, 'time_block', None)
    return C


def requests_of(C, unit):
    return [({'segment_id': s, 'target_date': d} if unit == 'daily' else {'segment_id': s, 'target_date': d, 'time_block': b})
            for s, d, b in zip(C['segment_id'], C['target_date'], C['time_block'])]


def predict_cells(N22, service, unit, C, chunk):
    """The accepted runtime on chunks <= chunk rows; predict_batch and predict_baseline_batch get the SAME ordered request list. Any error response
    fails closed (no partial or zero answer). Returns per-cell selected/baseline lambdas + routing/source availability."""
    if not 1 <= int(chunk) <= int(service['meta']['max_batch']):
        raise EvalError('chunk must be 1..max_batch')
    reqs, rows = requests_of(C, unit), []
    for i in range(0, len(reqs), int(chunk)):
        rq = reqs[i:i + int(chunk)]
        a, b = N22['predict_batch'](service, unit, rq), N22['predict_baseline_batch'](service, unit, rq)
        for r in (a, b):
            if not r.get('ok'):
                raise EvalError('runtime refused a 2026 chunk at ' + str(i) + ': ' + json.dumps(r.get('error'), ensure_ascii=False)[:400])
        for q, x, y in zip(rq, a['results'], b['results']):
            kx = (x['segment_id'], x['target_date'], x['time_block'])
            ky = (y['segment_id'], y['target_date'], y['time_block'])
            kq = (q['segment_id'], q['target_date'], q.get('time_block'))
            if not (kx == ky == kq):
                raise EvalError('runtime response keys differ from the identical ordered request keys')
            sa = x['source_availability']
            rows.append({'pred_selected': x['expected_recorded_count'], 'pred_baseline': y['expected_recorded_count'], 'component_used': x['component_used'],
                         'routing_reason': x['routing_reason'], 'baseline_method': y['method'], 'selected_method': x['method'],
                         'model_version': x['model_version'], 'issue_time_local': x['issue_time_local'],
                         'aadt_source': sa.get('aadt'), 'weather_rain_source': sa.get('weather_daily_rain'), 'bank_calendar': sa.get('bank_calendar')})
        if len(a['results']) != len(rq) or len(b['results']) != len(rq):
            raise EvalError('runtime returned a different number of rows than requested')
    P = pd.concat([C.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    if not (np.isfinite(P[['pred_selected', 'pred_baseline']].to_numpy(float)).all() and (P[['pred_selected', 'pred_baseline']].to_numpy(float) >= 0).all()):
        raise EvalError('non-finite or negative runtime prediction')
    return P


def add_groups(P, cfg):
    cal = cfg['calendar']
    d = pd.to_datetime(P['target_date'], format='%Y-%m-%d')
    m2s = {m: s for s, ms in cal['seasons'].items() for m in ms}
    P = P.copy()
    P['month'] = d.dt.strftime('%Y-%m')
    P['iso_week_start'] = (d - pd.to_timedelta(d.dt.weekday, unit='D')).dt.strftime('%Y-%m-%d')
    P['weekday'] = [WEEKDAYS[w] for w in d.dt.weekday]
    P['season'] = d.dt.month.map(m2s).fillna(UNKNOWN)
    fest = np.full(len(P), 'none', dtype=object)
    for name, (a, b) in cal['festivals'].items():
        fest[((P['target_date'] >= a) & (P['target_date'] <= b)).to_numpy()] = name
    P['festival'] = fest
    P['time_block_group'] = P['time_block'].fillna('day')
    P['missing_context'] = np.where(P['routing_reason'] == 'context_complete', 'context_complete', 'core_fallback_missing_context')
    return P


GROUP_DIMS = ('overall', 'month', 'iso_week_start', 'weekday', 'time_block_group', 'season', 'festival', 'province', 'route', 'missing_context', 'component_used')


DEVIANCE_STATUSES = ('finite', 'infinite_due_zero_prediction', 'unavailable_empty')


def _deviance_terms(y, m):
    """Unit Poisson deviance 2*(y*log(y/m) - (y - m)); y*log(y/m) := 0 for y == 0. m must be > 0 wherever y > 0."""
    with np.errstate(divide='ignore', invalid='ignore'):
        t = np.where(y > 0, y * np.log(np.where(y > 0, y, 1.0) / np.where(m > 0, m, 1.0)), 0.0)
    return 2.0 * (t - (y - m))


def metric_row(y, p, floor=None):
    """Raw metrics on unmodified lambdas. Raw mean Poisson deviance: finite when every positive cell has p > 0; a zero prediction on a positive cell
    makes the limiting raw deviance +infinity -> value null (strict JSON) + deviance_status 'infinite_due_zero_prediction' + the count.
    floor (the original locked notebook-20 numerical floor, audit only): m = max(p, floor) inside the deviance ONLY; floored count and the floored
    rows' contribution to the mean are reported. Never used to select, tune or change eligibility."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    n, sy = len(y), float(y.sum())
    zero_pos = int(((p <= 0) & (y > 0)).sum())
    status = DEVIANCE_STATUSES[2] if not n else (DEVIANCE_STATUSES[1] if zero_pos else DEVIANCE_STATUSES[0])
    dev = float(np.mean(_deviance_terms(y, p))) if status == 'finite' else None
    e = p - y
    out = {'n_positive_cells': int(n), 'n_raw_records': int(sy), 'actual_total': sy, 'pred_total': float(p.sum()),
           'mean_poisson_deviance': dev, 'deviance_status': status, 'n_zero_pred_on_positive': zero_pos, 'mae': float(np.abs(e).mean()) if n else None,
           'rmse': float(np.sqrt((e ** 2).mean())) if n else None, 'bias_pred_minus_actual': float(e.mean()) if n else None,
           'wape': float(np.abs(e).sum() / sy) if sy > 0 else None, 'pred_to_actual_ratio': float(p.sum() / sy) if sy > 0 else None}
    if floor is not None:
        fl = float(floor)
        d = _deviance_terms(y, np.maximum(p, fl))
        low = p < fl
        out.update({'deviance_floor_audit': float(d.mean()) if n else None, 'deviance_floor_audit_floor': fl,
                    'deviance_floor_audit_n_floored': int(low.sum()), 'deviance_floor_audit_floored_contribution': float(d[low].sum() / n) if n else None,
                    'deviance_floor_audit_floored_share': float(d[low].sum() / d.sum()) if n and d.sum() > 0 else None})
    return out


def group_metrics(P, unit, floor=None):
    rows = []
    for dim in GROUP_DIMS:
        groups = [('all', P.index)] if dim == 'overall' else list(P.groupby(dim, dropna=False).groups.items())
        for g, idx in groups:
            sub = P.loc[idx]
            base = {'unit': unit, 'dim': dim, 'group': str(g), 'n_segments': int(sub['segment_id'].nunique())}
            for meth, col in (('selected', 'pred_selected'), ('strong_baseline', 'pred_baseline')):
                rows.append(dict(base, method=meth, **metric_row(sub['y'], sub[col], floor)))
    return pd.DataFrame(rows)


def compare_methods(M, min_support):
    """Selected vs baseline on the SAME cells per group; baseline-better groups are reported, never hidden. No eligibility change."""
    k = ['unit', 'dim', 'group']
    s = M[M['method'] == 'selected'].set_index(k)
    b = M[M['method'] == 'strong_baseline'].set_index(k)
    out = pd.DataFrame({'n_positive_cells': s['n_positive_cells'], 'n_raw_records': s['n_raw_records'], 'n_segments': s['n_segments'],
                        'mae_selected': s['mae'], 'mae_baseline': b['mae'], 'deviance_selected': s['mean_poisson_deviance'],
                        'deviance_baseline': b['mean_poisson_deviance'], 'deviance_status_selected': s['deviance_status'],
                        'deviance_status_baseline': b['deviance_status'], 'n_zero_pred_on_positive_selected': s['n_zero_pred_on_positive'],
                        'n_zero_pred_on_positive_baseline': b['n_zero_pred_on_positive'], 'ratio_selected': s['pred_to_actual_ratio'],
                        'ratio_baseline': b['pred_to_actual_ratio']})
    if 'deviance_floor_audit' in M.columns:
        out['deviance_floor_audit_selected'], out['deviance_floor_audit_baseline'] = s['deviance_floor_audit'], b['deviance_floor_audit']
    out['supported'] = out['n_positive_cells'] >= int(min_support)
    out['better_mae'] = np.where(out['mae_selected'] < out['mae_baseline'], 'selected', np.where(out['mae_selected'] > out['mae_baseline'], 'strong_baseline', 'tie'))
    ds, db = out['deviance_selected'].astype(float), out['deviance_baseline'].astype(float)
    out['better_deviance'] = np.where(ds.isna() | db.isna(), 'undefined', np.where(ds < db, 'selected', np.where(ds > db, 'strong_baseline', 'tie')))
    return out.reset_index()


RANK_KINDS = ('under_predicted', 'over_predicted', 'total_match')


def under_over_ranking(M, min_support, top_n):
    """TRUE direction only: under_predicted = pred/actual ratio < 1 (lowest first), over_predicted = ratio > 1 (highest first), total_match = ratio == 1.
    Supported non-overall groups only. An empty under/over side gets ONE explicit 'none' row (rank/ratio null) carrying the supported-group counts,
    never the other side's rows relabelled. Descriptive only; never used to change eligibility or selection."""
    s = M[(M['dim'] != 'overall') & (M['n_positive_cells'] >= int(min_support))].copy()
    s = s[s['pred_to_actual_ratio'].notna()]
    rows = []
    for u in sorted(M['unit'].unique()):
        for meth in sorted(M['method'].unique()):
            g = s[(s['unit'] == u) & (s['method'] == meth)]
            r_ = g['pred_to_actual_ratio'].astype(float)
            parts = {'under_predicted': g[r_ < 1].sort_values(['pred_to_actual_ratio', 'dim', 'group']),
                     'over_predicted': g[r_ > 1].sort_values(['pred_to_actual_ratio', 'dim', 'group'], ascending=[False, True, True]),
                     'total_match': g[r_ == 1].sort_values(['dim', 'group'])}
            cnt = {'n_supported_groups': int(len(g)), 'n_groups_under': int(len(parts['under_predicted'])),
                   'n_groups_over': int(len(parts['over_predicted'])), 'n_groups_total_match': int(len(parts['total_match']))}
            for kind in RANK_KINDS:
                part = parts[kind] if kind == 'total_match' else parts[kind].head(int(top_n))
                if not len(part) and kind != 'total_match':
                    rows.append(dict({'unit': u, 'method': meth, 'kind': kind, 'rank': None, 'dim': 'none', 'group': 'NONE', 'pred_to_actual_ratio': None,
                                      'n_positive_cells': None, 'n_raw_records': None,
                                      'note': 'no supported group with pred/actual ratio ' + ('< 1' if kind == 'under_predicted' else '> 1')}, **cnt))
                for rank, r in enumerate(part.itertuples(index=False), 1):
                    rows.append(dict({'unit': u, 'method': meth, 'kind': kind, 'rank': rank, 'dim': r.dim, 'group': r.group,
                                      'pred_to_actual_ratio': r.pred_to_actual_ratio, 'n_positive_cells': r.n_positive_cells, 'n_raw_records': r.n_raw_records,
                                      'note': 'descriptive only; never used to change eligibility'}, **cnt))
    return pd.DataFrame(rows)


def multi_province_counts(C_by_unit):
    """Cell province = mode of the recorded event provinces of its records (ties -> lexically first); NOT the OOF segment-midpoint province."""
    return pd.DataFrame([{'unit': u, 'positive_cells': int(len(C)), 'multi_province_cells': int((C['n_provinces'] > 1).sum()),
                          'raw_records_in_multi_province_cells': int(C.loc[C['n_provinces'] > 1, 'y'].sum()),
                          'province_rule': 'mode of recorded event province per cell, ties lexical; not the segment-midpoint province used by OOF error groups'}
                         for u, C in C_by_unit.items()])


def coverage_flags(window, n_segments, C_by_unit, n_blocks):
    """Requested calendar x segment cells vs observed positive cells per month; every unobserved cell stays UNKNOWN (zero label unverified)."""
    days = pd.date_range(window['start'], window['end'], freq='D').strftime('%Y-%m-%d')
    rows = []
    for unit, C in C_by_unit.items():
        per = 1 if unit == 'daily' else int(n_blocks)
        obs = C.groupby(C['target_date'].str[:7]).size()
        for m, ds in pd.Series(days).groupby(pd.Series(days).str[:7]):
            req = int(len(ds) * n_segments * per)
            o = int(obs.get(m, 0))
            rows.append({'unit': unit, 'month': m, 'requested_cells': req, 'observed_positive_cells': o, 'unknown_cells': req - o,
                         'zero_label_status': 'unverified_unknown_not_zero', 'full_population_metric': 'not_available'})
    return pd.DataFrame(rows)


In [ ]:
# FUNCTIONS (4/4): aggregate figures (PNG + SVG), the Thai ModelGate report and the strict manifest. Reports hold aggregates only (no raw rows,
# no record IDs, no coordinates). Every figure is drawn from the same aggregate tables written as CSV.
def setup_fonts(names):
    import matplotlib
    from matplotlib import font_manager
    have = {f.name for f in font_manager.fontManager.ttflist}
    use = [n for n in names if n in have]
    matplotlib.rcParams['font.family'] = use + ['DejaVu Sans'] if use else ['DejaVu Sans']
    matplotlib.rcParams['svg.hashsalt'] = 'roadaware-2026'
    return use


def save_figure(fig, stem, report_dir):
    """Deterministic PNG + SVG (no timestamps in metadata). Returns {file: bytes}."""
    out = {}
    for ext, kw in (('.png', {'dpi': 130, 'metadata': {'Software': None}}), ('.svg', {'metadata': {'Date': None, 'Creator': None}})):
        buf = io.BytesIO()
        fig.savefig(buf, format=ext[1:], bbox_inches='tight', **kw)
        out[stem + ext] = buf.getvalue()
    import matplotlib.pyplot as plt
    plt.close(fig)
    return out


def make_figures(T, fonts):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    setup_fonts(fonts)
    figs = {}
    # 1 monthly exclusive ledger
    led = T['ledger_by_month'].set_index('event_month')
    cols = [c for c in EXCLUSIVE_REASONS if c in led.columns and led[c].sum() > 0]
    fig, ax = plt.subplots(figsize=(11, 5))
    bottom = np.zeros(len(led))
    for c in cols:
        ax.bar(led.index.astype(str), led[c].to_numpy(float), bottom=bottom, label=c)
        bottom += led[c].to_numpy(float)
    ax.set_title('2026 raw observed records by event month: exclusive eligibility ledger')
    ax.set_ylabel('raw observed records')
    ax.legend(fontsize=7, ncol=2)
    ax.tick_params(axis='x', rotation=45)
    figs.update(save_figure(fig, 'fig_ledger_by_month', None))
    # 2 old vs new snapshot monthly counts
    on = T['old_new_monthly']
    fig, ax = plt.subplots(figsize=(10, 4.5))
    x = np.arange(len(on))
    ax.bar(x - 0.2, on['old_raw_records'], 0.4, label='original local CSV (ended 2026-09-07)')
    ax.bar(x + 0.2, on['new_raw_records'], 0.4, label='CKAN snapshot 2026-11-02')
    ax.set_xticks(x, on['event_month'].astype(str), rotation=45)
    ax.set_title('Net raw record-count change by event month (not proven unique late events)')
    ax.legend(fontsize=8)
    figs.update(save_figure(fig, 'fig_old_new_monthly', None))
    # 3 report lag by month
    lg = T['lag_by_month']
    lg = lg[lg['event_month'] != UNKNOWN]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for c in ('lag_median_days', 'lag_p90_days', 'lag_p95_days', 'lag_p99_days'):
        ax.plot(lg['event_month'].astype(str), lg[c].astype(float), marker='o', label=c)
    ax.axhline(30, color='grey', linestyle='--', label='30-day research label assumption')
    ax.set_ylabel('report date - event date (days, parseable only)')
    ax.set_title('Report lag by event month (later months right-truncated at the snapshot)')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)
    figs.update(save_figure(fig, 'fig_report_lag_by_month', None))
    # 4/5 per unit: monthly MAE and totals, selected vs baseline on identical cells
    for unit in (UNITS if T.get('metrics') is not None else ()):
        M = T['metrics'][(T['metrics']['unit'] == unit) & (T['metrics']['dim'] == 'month')]
        if not len(M):
            continue
        fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
        for meth in ('selected', 'strong_baseline'):
            g = M[M['method'] == meth].sort_values('group')
            axs[0].plot(g['group'], g['mae'].astype(float), marker='o', label=meth)
            axs[1].plot(g['group'], g['pred_total'].astype(float), marker='o', label=meth + ' predicted total')
        g = M[M['method'] == 'selected'].sort_values('group')
        axs[1].plot(g['group'], g['actual_total'].astype(float), marker='s', color='black', label='observed total (positive cells only)')
        axs[0].set_title(unit + ': MAE on observed positive cells (outcome-conditioned)')
        axs[1].set_title(unit + ': totals on observed positive cells')
        axs[0].set_ylabel('MAE (raw records per observed positive cell)')
        axs[1].set_ylabel('raw records (sum over observed positive cells)')
        for a in axs:
            a.legend(fontsize=8)
            a.tick_params(axis='x', rotation=45)
        figs.update(save_figure(fig, 'fig_metrics_by_month_' + unit, None))
    return figs


def _cell(v):
    if v is None or (isinstance(v, float) and not math.isfinite(v)):
        return 'n/a'
    if isinstance(v, float):
        return format(v, '.4g')
    return str(v).replace('|', '/')


def md_table(df, max_rows=40):
    if df is None or not len(df):
        return '(ไม่มีข้อมูล)' + NL
    d = df.head(max_rows)
    lines = ['| ' + ' | '.join(map(str, d.columns)) + ' |', '|' + '---|' * len(d.columns)]
    lines += ['| ' + ' | '.join(_cell(v) for v in r) + ' |' for r in d.itertuples(index=False)]
    return NL.join(lines) + NL


MAIN_METRIC_COLUMNS = ('unit', 'method', 'n_positive_cells', 'n_raw_records', 'n_segments', 'actual_total', 'pred_total', 'mean_poisson_deviance', 'deviance_status',
                       'n_zero_pred_on_positive', 'deviance_floor_audit', 'deviance_floor_audit_n_floored', 'deviance_floor_audit_floored_contribution', 'mae',
                       'rmse', 'bias_pred_minus_actual', 'wape')


def ranking_summary_th(RK):
    """One honest line per unit x method: counts of supported groups that are truly under (ratio < 1) / over (ratio > 1)."""
    if RK is None or not len(RK):
        return ['- (ไม่มีกลุ่มที่มี support พอสำหรับจัดอันดับ)']
    out = []
    for (u, meth), g in RK.groupby(['unit', 'method'], sort=True):
        r = g.iloc[0]
        line = ('- ' + str(u) + ' / ' + str(meth) + ': กลุ่มที่มี support ' + str(int(r['n_supported_groups'])) + ' กลุ่ม; ทำนายต่ำกว่าจริง (ratio < 1) ' +
                str(int(r['n_groups_under'])) + ' กลุ่ม, ทำนายสูงกว่าจริง (ratio > 1) ' + str(int(r['n_groups_over'])) + ' กลุ่ม')
        if int(r['n_groups_over']) == 0:
            line += ' — **ไม่มีกลุ่มใดที่ทำนายสูงกว่าจริง**; กลุ่มที่ ratio สูงสุดก็ยังทำนายต่ำกว่าจริง'
        if int(r['n_groups_under']) == 0:
            line += ' — **ไม่มีกลุ่มใดที่ทำนายต่ำกว่าจริง**'
        out.append(line)
    return out


def report_th(status, T, facts, cfg):
    ov = T['metrics'][T['metrics']['dim'] == 'overall'] if T.get('metrics') is not None else None
    fl = facts.get('deviance_floor') or {}
    s = ['# รายงานวินิจฉัยข้อมูลปี 2026 (raw observed records) — RoadAware V3', '',
         '**สถานะ:** ' + status + ' | research only | production_ready = false', '',
         '## 1. ข้อเท็จจริงของแหล่งข้อมูล', '',
         '- ไฟล์ CKAN resource ' + cfg['source']['resource_id'] + ' last_modified ' + str(facts['ckan']['last_modified']) + ', ' + str(facts['ckan']['file_bytes']) +
         ' bytes, raw rows ' + str(facts['new_dates']['raw_rows']) + ' แถว (1 แถว = 1 raw observed record; ไม่มี ACC_CODE จึงยืนยัน unique physical event ไม่ได้)',
         '- sha256 ของไฟล์ ' + str(facts.get('source_sha256')) + ' (ตรงกับ manager source pin); ckan hash เท่ากับ md5 ของไฟล์: ' +
         str(facts['ckan'].get('ckan_hash_equals_file_md5')),
         '- วันที่เกิดเหตุ ' + str(facts['new_dates']['event_date_min']) + ' ถึง ' + str(facts['new_dates']['event_date_max']) + '; วันที่รายงาน ' +
         str(facts['new_dates']['report_date_min']) + ' ถึง ' + str(facts['new_dates']['report_date_max']) + '; exact full-row duplicates หลังแถวแรก ' +
         str(facts['new_dates']['exact_full_row_duplicates_after_first']),
         '- ช่วงที่ขอ ' + cfg['window']['start'] + '..' + cfg['window']['end'] + ' (ตัดตุลาคมด้วยกติกาวันที่ก่อนดูคะแนน). ไฟล์เดิมจบ 7 ก.ย. จึงไม่ใช้อ้างครบกันยายน',
         '- วันอัปเดตไฟล์ไม่ใช่หลักฐานว่าข้อมูลครบ; ข้อมูลที่เข้ามาหลัง 2026-11-02 ไม่ทราบ', '',
         '## 2. Ledger การคัดเลือก (exclusive ตามลำดับที่ลงทะเบียนไว้)', '', md_table(T['ledger_order']), '',
         'รวมทุกเหตุผลเท่ากับ raw rows ทุกแถว (reconciled). ตารางรายเดือน/จังหวัด/หน่วยงานอยู่ใน ledger_by_*.csv (UNKNOWN คงไว้)', '',
         '## 3. ความล่าช้าในการรายงานและความครบถ้วน', '', md_table(T['lag_by_month'][['event_month', 'n_raw_records', 'n_lag_known', 'n_report_date_unknown',
                                                                                'n_negative_lag', 'lag_median_days', 'lag_p90_days', 'lag_p99_days',
                                                                                'share_lag_gt_label_days', 'late_arrival_risk_indicator', 'maturity']]), '',
         '- lag = วันที่รายงาน − วันที่เกิดเหตุ (วัน) เฉพาะ record ที่ parse ได้ทั้งสองวันและ lag ≥ 0; report date ที่ parse ไม่ได้คงเป็น unknown (null) และ lag ติดลบนับแยกไว้ ไม่ถูกนับเป็น 0',
         '- maturity = unverified ทุกเดือน: ไม่มีหลักฐานว่าเดือนใดครบแล้ว และ CKAN last_modified ไม่ได้พิสูจน์ความครบถ้วน',
         '- สมมติฐาน label พร้อมใช้ใน 30 วันของโปรโตคอลเดิมเป็น research assumption; lag ที่วัดได้ (share_lag_gt_label_days > 0) ขัดแย้งกับการอ้างว่าครบใน 30 วัน (ไม่ได้แก้โปรโตคอลเดิมเงียบ ๆ)',
         '- **นิยาม late_arrival_risk_indicator:** สำหรับแต่ละ record ในกลุ่มที่วันที่เกิดเหตุ parse ได้ ให้ อายุ = snapshot (' + str(cfg['source']['snapshot_report_last_date']) +
         ') − วันที่เกิดเหตุ (วัน); คำนวณสัดส่วนของ lag เชิงประจักษ์ของเดือนอ้างอิง ' + str(cfg['metrics']['lag_reference_months']) +
         ' (ม.ค.–มี.ค., เฉพาะ lag ที่ ok) ที่ **มากกว่า** อายุนั้น แล้วเฉลี่ยทุก record ในกลุ่ม',
         '- ตัวชี้นี้เป็น risk indicator เท่านั้น: การกระจาย lag อ้างอิงเองก็ถูกตัดขวา (right-truncated) ณ snapshot; **ไม่ใช่** สัดส่วนเหตุการณ์จริงที่ยังไม่ถูกรายงาน ไม่ใช่ความน่าจะเป็น ไม่ใช่ค่าประมาณความครบถ้วน และไม่ได้ใช้ถ่วงน้ำหนักหรือแก้ค่าใด ๆ', '',
         '## 4. การเปรียบเทียบ snapshot เดิมกับใหม่ (net record-count change/revisions)', '', md_table(T['old_new_monthly'].drop(columns=['interpretation'])), '',
         'ไม่มี ACC_CODE: แถวที่เปลี่ยนค่าในบางฟิลด์กับแถวใหม่แยกกันไม่ได้ จึงไม่ใช่หลักฐาน late-arriving unique events', '',
         '## 5. ผลวินิจฉัยโมเดลบน observed positive cells เท่านั้น', '']
    if ov is not None and len(ov):
        s += [md_table(ov[[c for c in MAIN_METRIC_COLUMNS if c in ov.columns]]), '',
              '- ประเมินเฉพาะ cell (segment × วัน หรือ × ช่วง 6 ชม.) ที่มี eligible linked record อย่างน้อย 1 รายการ ทั้ง selected HGB policy และ strong baseline ใช้ key/y ชุดเดียวกัน',
              '- **outcome-conditioned selection bias:** ทุก cell มี y ≥ 1 จึงไม่เทียบโดยตรงกับ full-population OOF 2022–23 หรือ seen diagnostic 2024–25; ไม่มี verified zero และไม่มี full-population 2026 accuracy',
              '- actual_total/pred_total เป็นผลรวมบน observed positive cells เท่านั้น ไม่ใช่การพยากรณ์ทั้งประชากรหรือรายปี', '',
              '### นิยาม deviance', '',
              '- **mean_poisson_deviance (raw, metric หลักของรายงานนี้):** (1/N) Σ 2[y·ln(y/λ) − (y − λ)] บน λ ที่ไม่ปรับค่า. ถ้ามี cell ที่ y > 0 แต่ λ = 0 ค่า limit คือ +∞ '
              '→ แสดง n/a (JSON null เพราะ strict JSON ไม่รองรับ Infinity) พร้อม deviance_status = infinite_due_zero_prediction และจำนวน n_zero_pred_on_positive; '
              'finite = คำนวณได้, unavailable_empty = ไม่มี cell',
              '- **deviance_floor_audit (numerical-floor audit เท่านั้น):** สูตรเดิมของ notebook 20 ที่ล็อกไว้ m = max(λ, ' + repr(fl.get('floor')) + ') เฉพาะใน metric '
              '(floor จาก config20 sha256 ' + str(fl.get('config20_sha256'))[:12] + ' ผ่าน runtime pin chain, ไม่ได้เปลี่ยนค่า). deviance_floor_audit_n_floored = จำนวน cell ที่ λ < floor, '
              'deviance_floor_audit_floored_contribution = ส่วนของ mean ที่มาจาก cell เหล่านั้น. ค่านี้ใช้ตรวจสอบเท่านั้น **ไม่ใช่เกณฑ์ใหม่ ไม่ได้ใช้เลือก/ปรับโมเดล** '
              'และไม่ได้ทำให้ raw deviance ของ baseline กลายเป็นค่าจำกัด; ถ้า n_floored = 0 นิยามทั้งสองให้ค่าเดียวกันบน cohort นี้', '',
              '### กลุ่ม under/over (ทิศทางจริง)', ''] + ranking_summary_th(T.get('under_over_ranking')) + [
              '- under_over_ranking.csv: kind = under_predicted (ratio < 1, ต่ำสุดก่อน) / over_predicted (ratio > 1, สูงสุดก่อน) / total_match; ฝั่งที่ว่างมีแถว NONE ระบุชัด',
              '- กลุ่มที่ baseline ดีกว่ารายงานไว้ใน method_comparison.csv; การจัดอันดับเป็นคำอธิบายเท่านั้น ไม่ได้ใช้เปลี่ยน eligibility หรือการเลือกโมเดล', '',
              '### นิยามกลุ่มภูมิศาสตร์และปฏิทิน', '',
              '- จังหวัดของ cell ปี 2026 = จังหวัดที่บันทึกใน record ที่พบบ่อยที่สุดของ cell (เสมอกันเลือกตามลำดับตัวอักษร) **ไม่ใช่** จังหวัดจากจุดกึ่งกลาง segment ที่ใช้ใน OOF error groups จึงเทียบกลุ่มโดยตรงไม่ได้',
              md_table(T.get('multi_province')), '',
              '- ช่วงเทศกาล ' + ', '.join(k + ' ' + v[0] + '..' + v[1] for k, v in cfg['calendar']['festivals'].items()) + ' และฤดู (ตามเดือน) เป็น analytical windows สำหรับจัดกลุ่มเท่านั้น '
              'ไม่ใช่วันหยุดราชการที่ตรวจสอบแล้ว และไม่ใช่ input ของโมเดล. ' + cfg['calendar'].get('timeline_note', ''), '']
    else:
        s += ['**diagnostic unavailable:** ' + str(facts.get('unavailable_reason')), '']
    s += ['## 6. แยกหลักฐานสามชุด', '', '- OOF 2022–23: ใช้เลือกโมเดล (ก่อนมีคะแนน 2026)', '- 2024–25: previously seen diagnostic',
          '- 2026: raw observed-record positive-cell diagnostic (ไม่ใช่ strict untouched test เพราะเคยมี data audit ปี 2026 ในโครงการ)', '',
          '## ข้อจำกัด', ''] + ['- ' + c for c in cfg['caveats']] + ['']
    return NL.join(s)


def build_manifest(status, started, cfg, cfg_sha, pins_sha, checks, report_files, data_files, protected_before, protected_after, extra):
    return {'kind': 'roadaware_2026_record_diagnostics', 'version': cfg['version'], 'status': status, 'started_utc': started, 'finished_utc': utcnow(),
            'research_only': True, 'production_ready': False, 'record_meaning': cfg['record_meaning'], 'config_sha256': cfg_sha, 'pins_sha256': pins_sha,
            'checks': checks, 'checks_passed': sum(c['status'] == 'pass' for c in checks), 'checks_failed': sum(c['status'] != 'pass' for c in checks),
            'report_files': report_files, 'data_files_gitignored': data_files, 'protected_before': protected_before, 'protected_after': protected_after,
            'protected_unchanged': protected_before == protected_after, 'environment': environment(), 'caveats': cfg['caveats'], **extra}


In [ ]:
# SETUP: config, dependency check, NEW output root guard (before any write), protected snapshot, check/report helpers.
import contextlib

STARTED = utcnow()
cfg = read_strict_json(CONFIG_PATH)
CFG_SHA = sha256_file(CONFIG_PATH)
VERSIONS = require_modules(cfg['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, cfg)
check_outputs(PATHS, BACKEND, cfg)
PROTECTED_BEFORE = hash_paths(BACKEND, sorted(set(cfg['protected_paths'])))
PATHS['report_dir'].mkdir(parents=True)
PATHS['data_dir'].mkdir(parents=True)
CHECKS, DURATIONS, REPORT_FILES, DATA_FILES, T, FACTS = [], {}, {}, {}, {}, {}


def check(name, ok, evidence=None):
    CHECKS.append({'check': name, 'status': 'pass' if ok else 'fail', 'evidence': jsonable(evidence)})
    if not ok:
        raise EvalError('check failed: ' + name + ' | ' + json.dumps(jsonable(evidence), allow_nan=False, ensure_ascii=False)[:600])


@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    print('stage', name, '...', flush=True)
    yield
    DURATIONS[name] = round(time.time() - t0, 2)
    print('stage', name, 'done in', DURATIONS[name], 's', flush=True)


def put_report(name, obj):
    check_report_file(name, cfg)
    b = csv_bytes(obj) if isinstance(obj, pd.DataFrame) else (obj if isinstance(obj, bytes) else strict_json_bytes(obj))
    atomic_write_bytes(PATHS['report_dir'] / name, b)
    REPORT_FILES[name] = sha256_bytes(b)


def put_data(name, df):
    b = csv_bytes(df)
    atomic_write_bytes(PATHS['data_dir'] / name, b)
    DATA_FILES[name] = sha256_bytes(b)


print('versions', VERSIONS, '| report dir', PATHS['report_dir'], '| data dir', PATHS['data_dir'])


In [ ]:
# SOURCE AUDIT: manager source + linkage pins verified -> CKAN provenance -> exact schema -> explicit D/M/Y dates, times, lag/maturity,
# exact full-row duplicates and the aggregate old/new snapshot comparison. Raw snapshot and original are read only.
with stage('source_audit'):
    SRC_PINS, SRC_PINS_SHA = load_pins(BACKEND, cfg, 'source')
    LNK_PINS, LNK_PINS_SHA = load_pins(BACKEND, cfg, 'linkage')
    INPUT_ROWS = verify_source_pins(BACKEND, cfg, SRC_PINS) + verify_linkage_pins(BACKEND, cfg, LNK_PINS)
    check('manager source and linkage pins: every pinned file hash-verified before any parsing or FUNCTIONS exec', True,
          {'verified_files': len(INPUT_ROWS), 'source_pins_sha256': SRC_PINS_SHA, 'linkage_pins_sha256': LNK_PINS_SHA})
    S, NS = cfg['source'], SRC_PINS['new_snapshot']
    FACTS['ckan'] = ckan_provenance(read_strict_json(BACKEND / S['ckan']), S['resource_id'], BACKEND / S['new_csv'], NS)
    check('saved CKAN resource matches the pinned snapshot (format CSV, size == file bytes, last_modified, resource URL)', True, FACTS['ckan'])
    FACTS['source_sha256'] = sha256_file(BACKEND / S['new_csv'])
    check('new snapshot sha256 equals the manager source pin', FACTS['source_sha256'] == SRC_PINS['files'][pin_key(cfg, S['new_csv'])], FACTS['source_sha256'])
    HEADER, RAW = read_raw_csv(BACKEND / S['new_csv'], S['encoding'], S['expected_columns'])
    OHEADER, ORAW = read_raw_csv(BACKEND / S['original_csv'], S['encoding'], S['expected_columns'])
    check('new snapshot schema: 38 unique columns incl. every configured field; no ACC_CODE column; original has the identical header',
          all(c in HEADER for c in S['columns'].values()) and not any('ACC' in h.upper() for h in HEADER) and OHEADER == HEADER,
          {'columns': len(HEADER), 'rows': len(RAW), 'original_rows': len(ORAW)})
    check('new snapshot raw row count equals the manager pin', len(RAW) == int(NS['rows']), {'rows': len(RAW), 'pinned': NS['rows']})
    PROV = pd.read_csv(BACKEND / cfg['linkage']['province_points'], dtype=str, keep_default_na=False)
    PROVINCE_KEYS = set(PROV['province_key'].map(norm_text)) - {''}
    R = parse_records(RAW, HEADER, cfg, PROVINCE_KEYS)
    RO = parse_records(ORAW, OHEADER, cfg, PROVINCE_KEYS)
    FACTS['new_dates'], FACTS['old_dates'] = date_audit(R, cfg['window']), date_audit(RO, cfg['window'])
    check('event and report date ranges equal the pinned snapshot facts (event max, report max)',
          FACTS['new_dates']['event_date_max'] == NS['event_last_date'] and FACTS['new_dates']['report_date_max'] == NS['report_last_date'], FACTS['new_dates'])
    c16_path = BACKEND / cfg['coordinates']['readiness_config']
    c16 = read_strict_json(c16_path)
    cc, ca = cfg['coordinates'], c16['coordinate_audit']
    check('coordinate replica constants equal the accepted notebook-16 config (bbox, round_dp, low-precision decimals, repeated threshold)',
          all(float(c16['thailand_broad_bbox'][k]) == float(cc['thailand_broad_bbox'][k]) for k in cc['thailand_broad_bbox'])
          and all(int(ca[k]) == int(cc[k]) for k in ('round_dp', 'low_precision_max_decimals', 'repeated_min_events')), {'config16_sha256': sha256_file(c16_path)})
    COORDS = audit_coordinates(RAW, S['columns']['latitude'], S['columns']['longitude'], cc)
    LAG, LAG_ST = lag_days(R)
    R['lag_status'] = LAG_ST.to_numpy(object)
    M = cfg['metrics']
    T['lag_by_month'] = lag_table(R, LAG, LAG_ST, 'event_month', S['snapshot_report_last_date'], M['label_assumption_days'], M['lag_reference_months'])
    T['lag_by_province'] = lag_table(R, LAG, LAG_ST, 'province', S['snapshot_report_last_date'], M['label_assumption_days'], M['lag_reference_months'])
    T['old_new_monthly'] = old_new_comparison(RO, R)
    check('old/new comparison reconciles both files row-for-row', int(T['old_new_monthly']['old_raw_records'].sum()) == len(ORAW)
          and int(T['old_new_monthly']['new_raw_records'].sum()) == len(RAW), {'old': len(ORAW), 'new': len(RAW)})
    T['source_quality'] = pd.DataFrame([{'field': k, 'status': s, 'records': int(n)} for k in ('event_date_status', 'event_time_status', 'report_date_status',
                                                                                                  'report_time_status', 'lag_status')
                                        for s, n in R[k].value_counts().items()] +
                                       [{'field': 'coord_status', 'status': s, 'records': int(n)} for s, n in COORDS['coord_status'].value_counts().items()] +
                                       [{'field': 'province_valid', 'status': str(s), 'records': int(n)} for s, n in R['province_valid'].value_counts().items()] +
                                       [{'field': 'year_matches_date', 'status': str(s), 'records': int(n)} for s, n in R['year_matches_date'].value_counts().items()])
print(FACTS['new_dates'])
T['lag_by_month']


In [ ]:
# RUNTIME: manager runtime pins (22 source/config/manifest/record/input pins/baseline npz) verified -> pinned 22 FUNCTIONS (joined sha) exec'd ->
# accepted load_service (it re-verifies the whole 19/20/21 chain and every bundle hash BEFORE deserialization; startup self-test on). No fit.
with stage('runtime'):
    RT_PINS, RT_PINS_SHA = load_pins(BACKEND, cfg, 'runtime')
    RT_ROWS = verify_runtime_pins(BACKEND, cfg, RT_PINS)
    INPUT_ROWS += RT_ROWS
    RC = cfg['runtime']
    N22, N22_SHA = load_functions(BACKEND / RC['notebook22'], RC['functions_cells'], RC['functions_join'], RT_PINS['functions']['sha256'], 'notebook22')
    check('notebook-22 file, config, manifest, serving record, input pins and baseline npz equal the manager runtime pins; 22 FUNCTIONS joined sha equals the pin',
          True, {'verified_files': len(RT_ROWS), 'functions_sha256': N22_SHA, 'runtime_pins_sha256': RT_PINS_SHA})
    CFG22 = read_strict_json(BACKEND / RC['config22'])
    PINS22 = read_strict_json(BACKEND / CFG22['pins']['path'])
    BASELINE_FILES, SERVING_RECORD = baseline_files_from_record(BACKEND, cfg, RT_PINS)
    check('serving record pins/config sha equal the pinned input pins and config22', SERVING_RECORD['pins_sha256'] == sha256_file(BACKEND / CFG22['pins']['path'])
          and SERVING_RECORD['config_sha256'] == sha256_file(BACKEND / RC['config22']), {'record_model_version': SERVING_RECORD['model_version']})
    SERVICE = N22['load_service'](BACKEND, CFG22, PINS22, BASELINE_FILES, selftest=bool(RC['selftest']))
    HEALTH = N22['health'](SERVICE)
    check('accepted runtime available (research_ready, production_ready false), startup self-test passed', SERVICE['status'] == 'available'
          and HEALTH['production_ready'] is False, HEALTH)
    META = N22['metadata'](SERVICE)
    UNIVERSE = frozenset(SERVICE['seg_index'])
    N_BLOCKS = len(SERVICE['units']['6h']['blocks'])
    check('runtime model version equals the serving record; universe = qualified DOH segments', META['model_version'] == SERVING_RECORD['model_version']
          and len(UNIVERSE) == int(META['supported_segments']), {'model_version': META['model_version'], 'segments': len(UNIVERSE),
                                                                 'selected': {u: META['units'][u]['method'] for u in UNITS}, 'baseline': META['units']['daily']['baseline']})
HEALTH


In [ ]:
# LINKAGE + LEDGER: pinned notebook-18 FUNCTIONS (joined sha) -> accepted roads (derived files verified vs config18 AND the road manifest) ->
# 2026 records linked with the exact 50 m / 5 m rules -> independent segment-geometry re-check -> exclusive ordered ledger + multi-issue flags.
with stage('linkage_ledger'):
    L = cfg['linkage']
    N18, N18_SHA = load_functions(BACKEND / L['notebook18'], L['functions_cells'], L['functions_join'], LNK_PINS['notebook18_functions']['sha256'], 'notebook18')
    CFG18 = read_strict_json(BACKEND / L['config18'])
    N18['validate_config'](CFG18)
    check('notebook-18 FUNCTIONS joined sha equals the linkage pin; config18 hash-pinned; 50 m primary / 5 m margin unchanged', N18_SHA == LNK_PINS['notebook18_functions']['sha256']
          and float(CFG18['matching']['primary_tolerance_m']) == 50.0 and float(CFG18['matching']['ambiguity_margin_m']) == 5.0 and CFG18['alias_map'] == {},
          {'functions_sha256': N18_SHA})
    RM = build_road_model(N18, BACKEND, CFG18)
    check('road derived files verified against config18 and the accepted road manifest; overlap computation complete', RM['overlap_failed'] == 0,
          {'verified': len(RM['verify_rows']), 'road_counts': RM['road_counts']})
    E = link_records(N18, CFG18, RM, lineage_like(R, COORDS))
    check('linkage output aligned 1:1 with raw records', len(E) == len(R) and (E.index == R.index).all(), {'rows': len(E)})
    F = issue_flags(R, E, cfg['window'], cfg, UNIVERSE)
    REASON = exclusive_reason(F)
    ELIG = (REASON == 'eligible').to_numpy()
    check('exclusive ledger reconciles every raw record exactly once', int(REASON.value_counts().sum()) == len(RAW) and set(REASON) <= set(EXCLUSIVE_REASONS),
          REASON.value_counts().to_dict())
    REC = recheck_segment_distance(N18, CFG18, RM, E, ELIG, L['segment_column'], CFG18['matching']['piece_match_tol_m'], L['geometry_recheck_tol_m'])
    check('eligible records lie within 50 m of the geometry of their assigned qualified 2000 m segment (independent re-check, no snapping)',
          REC['failing'] == 0 and REC['missing_segment'] == 0 and REC['checked'] == int(ELIG.sum()), REC)
    check('eligible records: in window, DOH, verified route, matched at 50 m, assigned in-universe segment, valid date/time/block',
          bool(np.all(E[L['primary_status_column']].to_numpy(object)[ELIG] == 'matched')) and all(s in UNIVERSE for s in E[L['segment_column']].to_numpy(object)[ELIG])
          and all(b is not None for b in np.asarray(R['time_block'].to_numpy(object))[ELIG]), {'eligible': int(ELIG.sum())})
    TABS, T['ledger_order'], T['ledger_multi_issue'], T['ledger_cross'] = ledger_tables(R, F, REASON)
    T['ledger_by_month'], T['ledger_by_province'], T['ledger_by_agency'] = TABS['event_month'], TABS['province'], TABS['agency']
    for k in ('ledger_by_month', 'ledger_by_province', 'ledger_by_agency'):
        check(k + ' totals reconcile all raw records', int(T[k]['raw_records'].sum()) == len(RAW), {'sum': int(T[k]['raw_records'].sum())})
    # internal provenance id = file sha256 prefix + data row number; NOT an ACC_CODE / physical-event identity; never written to reports
    FILE_TAG = 'r2026:' + SRC_PINS['files'][pin_key(cfg, cfg['source']['new_csv'])][:12] + ':'
    UID = [FILE_TAG + str(int(r)) for r in R['source_row']]
    ROW_LEDGER = pd.DataFrame({'record_uid_internal': UID, 'source_row': R['source_row'], 'fingerprint': R['fingerprint'], 'event_date': R['event_date'],
                               'event_time_status': R['event_time_status'], 'time_block': R['time_block'], 'event_month': R['event_month'],
                               'report_date': R['report_date'], 'lag_status': R['lag_status'], 'province': R['province'], 'agency': R['agency'],
                               'source_id': E['source_id'], 'identity_status': E['identity_status'], 'route_key_matched': E['route_key_matched'],
                               'coord_status': E['coord_status'], 'precision_class': E['precision_class'], 'scope': E['scope'],
                               'status_50m': E[L['primary_status_column']], 'segment_status': E['segment_status'], 'segment_id_2000m': E[L['segment_column']],
                               'chosen_distance_m': E['chosen_distance_m'], 'in_universe': [s in UNIVERSE for s in E[L['segment_column']]], 'exclusive_reason': REASON})
    for r_ in ISSUE_FLAGS:
        ROW_LEDGER['flag_' + r_] = F[r_].to_numpy(bool)
print(T['ledger_order'].to_string())


In [ ]:
# PREDICTIONS + METRICS: observed POSITIVE eligible cells only (unobserved cells stay unknown, never zero) -> accepted runtime on identical ordered
# keys for the selected policy and the strong baseline (chunks <= 128) -> per-group metrics, method comparison, under/over ranking, coverage flags.
with stage('predictions_metrics'):
    MC = cfg['metrics']
    CELLS = {u: observed_cells(R, E, ELIG, cfg, u) for u in UNITS}
    for u in UNITS:
        check(u + ' observed positive cells: y >= 1 and sum(y) == eligible raw records', int(CELLS[u]['y'].sum()) == int(ELIG.sum()) and (CELLS[u]['y'] >= 1).all(),
              {'cells': len(CELLS[u]), 'records': int(CELLS[u]['y'].sum())})
    T['coverage_flags'] = coverage_flags(cfg['window'], len(UNIVERSE), CELLS, N_BLOCKS)
    T['multi_province'] = multi_province_counts(CELLS)
    # original locked notebook-20 numerical floor, read ONLY for the deviance audit column; config20 hash-verified against the pinned 22 input pins
    FA = MC['original_deviance_floor_audit']
    C20_REL = CFG22['inputs']['config20']
    C20_SHA = verify_file(BACKEND / C20_REL, PINS22['files'].get(pin_key(cfg, C20_REL)), 'config20 (via notebook-22 input pins)')
    FLOOR = float(read_strict_json(BACKEND / C20_REL)['metrics']['deviance_floor'])
    FACTS['deviance_floor'] = {'floor': FLOOR, 'config20': C20_REL, 'config20_sha256': C20_SHA, 'role': 'numerical-floor audit only; not a new criterion'}
    check('original notebook-20 deviance floor read from hash-verified config20 and equal to the registered value (audit only, unchanged)',
          FLOOR == float(FA['expected_floor']), FACTS['deviance_floor'])
    n_pos = min(len(CELLS[u]) for u in UNITS)
    EVAL_AVAILABLE = n_pos >= int(MC['min_positive_cells_for_evaluation'])
    FACTS['unavailable_reason'] = None if EVAL_AVAILABLE else ('only ' + str(n_pos) + ' observed positive eligible linked cells (< ' +
                                                               str(MC['min_positive_cells_for_evaluation']) + '); no scores fabricated')
    PRED, METRICS = {}, []
    if EVAL_AVAILABLE:
        for u in UNITS:
            PRED[u] = add_groups(predict_cells(N22, SERVICE, u, CELLS[u], cfg['runtime']['chunk_rows']), cfg)
            METRICS.append(group_metrics(PRED[u], u, FLOOR))
            check(u + ': selected and baseline scored on identical keys/targets; selected method = pinned selection; model version = serving record',
                  (PRED[u]['selected_method'] == META['units'][u]['method']).all() and (PRED[u]['baseline_method'] == META['units'][u]['baseline']).all()
                  and (PRED[u]['model_version'] == SERVING_RECORD['model_version']).all() and len(PRED[u]) == len(CELLS[u]),
                  {'rows': len(PRED[u]), 'components': PRED[u]['component_used'].value_counts().to_dict()})
        T['metrics'] = pd.concat(METRICS, ignore_index=True)
        T['method_comparison'] = compare_methods(T['metrics'], MC['min_group_support_cells'])
        T['under_over_ranking'] = under_over_ranking(T['metrics'], MC['min_group_support_cells'], MC['rank_top_n'])
        RK = T['under_over_ranking']
        rk_r = RK['pred_to_actual_ratio'].astype(float)
        check('under/over ranking uses true direction only (under ratio < 1, over ratio > 1, empty side explicit NONE)',
              bool((rk_r[RK['kind'] == 'under_predicted'].dropna() < 1).all() and (rk_r[RK['kind'] == 'over_predicted'].dropna() > 1).all()
                   and (RK.loc[rk_r.isna(), 'group'] == 'NONE').all()), RK.groupby(['unit', 'method', 'kind']).size().to_dict())
        check('raw deviance status consistent with zero-on-positive counts (null only when infinite/empty)',
              bool(((T['metrics']['deviance_status'] == 'finite') == T['metrics']['mean_poisson_deviance'].notna()).all()
                   and ((T['metrics']['deviance_status'] == 'infinite_due_zero_prediction') == (T['metrics']['n_zero_pred_on_positive'] > 0)).all()),
              T['metrics'].groupby(['unit', 'method', 'deviance_status']).size().to_dict())
        for u in UNITS:
            o = T['metrics'][(T['metrics']['unit'] == u) & (T['metrics']['dim'] == 'overall')]
            check(u + ' overall metrics reconcile with the cells (actual total == eligible records)', (o['actual_total'] == float(ELIG.sum())).all(), o.to_dict('records'))
    else:
        T['metrics'] = None
    STATUS = 'complete' if EVAL_AVAILABLE else 'complete_diagnostic_unavailable'
T['metrics'][T['metrics']['dim'] == 'overall'] if EVAL_AVAILABLE else FACTS['unavailable_reason']


In [ ]:
# REPORTS: aggregate CSV/PNG/SVG + Thai report in the report dir; row-level ledger and cell predictions only in the gitignored data dir;
# protected paths re-hashed; strict manifest written last.
with stage('reports'):
    for name in ('ledger_order', 'ledger_by_month', 'ledger_by_province', 'ledger_by_agency', 'ledger_multi_issue', 'ledger_cross', 'lag_by_month',
                 'lag_by_province', 'old_new_monthly', 'source_quality', 'coverage_flags', 'multi_province', 'metrics', 'method_comparison', 'under_over_ranking'):
        if T.get(name) is not None:
            put_report(name + '.csv', T[name])
    for fname, b in make_figures(T, cfg['figure_fonts']).items():
        put_report(fname, b)
    put_report('source_facts.json', {'ckan': FACTS['ckan'], 'source_sha256': FACTS['source_sha256'], 'deviance_floor_audit': FACTS.get('deviance_floor'),
                                     'snapshot_last_modified_proves_completeness': False,
                                     'new_snapshot_dates': FACTS['new_dates'], 'original_dates': FACTS['old_dates'],
                                     'requested_window': cfg['window'], 'evaluation_available': EVAL_AVAILABLE, 'unavailable_reason': FACTS['unavailable_reason'],
                                     'actual_support': {u: {'first_date': CELLS[u]['target_date'].min() if len(CELLS[u]) else None,
                                                            'last_date': CELLS[u]['target_date'].max() if len(CELLS[u]) else None,
                                                            'months_with_cells': sorted(set(CELLS[u]['target_date'].str[:7]))} for u in UNITS}})
    put_report('report_2026_th.md', report_th(STATUS, T, FACTS, cfg).encode('utf-8'))
    put_data('row_ledger.csv', ROW_LEDGER)
    for u in UNITS:
        put_data('cells_' + u + '.csv', CELLS[u])
        if u in PRED:
            put_data('predictions_' + u + '.csv', PRED[u])
    PROTECTED_AFTER = hash_paths(BACKEND, sorted(set(cfg['protected_paths'])))
    check('protected paths (accepted 14/16-22 notebooks, configs, roots, raw data, pins) unchanged', PROTECTED_BEFORE == PROTECTED_AFTER,
          {'paths': len(PROTECTED_BEFORE)})
    check('report dir holds only flat aggregate csv/md/json/png/svg files written by this run', sorted(p.name for p in PATHS['report_dir'].iterdir()) == sorted(REPORT_FILES),
          sorted(REPORT_FILES))
    MANIFEST = build_manifest(STATUS, STARTED, cfg, CFG_SHA, {'runtime': RT_PINS_SHA, 'source': SRC_PINS_SHA, 'linkage': LNK_PINS_SHA}, CHECKS, REPORT_FILES,
                              DATA_FILES, PROTECTED_BEFORE, PROTECTED_AFTER,
                              {'inputs_verified': INPUT_ROWS, 'functions_sha256': {'notebook22': N22_SHA, 'notebook18': N18_SHA},
                               'runtime': {'model_version': META['model_version'], 'selected': {u: META['units'][u]['method'] for u in UNITS},
                                           'baseline': META['units']['daily']['baseline'], 'bundle_sha256': {u: META['units'][u]['bundle_sha256'] for u in UNITS},
                                           'baseline_stats_sha256': {u: META['units'][u]['baseline_stats_sha256'] for u in UNITS},
                                           'experiment_lock_sha256': META['experiment_lock_sha256'], 'selection_sha256': META['selection_sha256']},
                               'counts': {'raw_records': len(RAW), 'eligible_records': int(ELIG.sum()),
                                          'positive_cells': {u: len(CELLS[u]) for u in UNITS}, 'reasons': REASON.value_counts().to_dict()},
                               'evaluation_available': EVAL_AVAILABLE, 'durations_seconds': DURATIONS, 'versions': VERSIONS})
    atomic_write_bytes(PATHS['report_dir'] / 'evaluation_2026_manifest.json', strict_json_bytes(MANIFEST))
print(STATUS, '| checks', len(CHECKS), '| reports', len(REPORT_FILES), '| data', len(DATA_FILES))


## Outputs (NEW root only, e.g. `backend/rerun-2026-eval-023-001`)

- `reports/v3/evaluation_2026/`: aggregate CSVs (ledger by order/month/province/agency, multi-issue, lag/maturity, old/new net change, source quality, coverage flags, multi-province cells, metrics with raw deviance status + original-floor audit, method comparison, true-direction under/over ranking), PNG + SVG figures, `source_facts.json`, `report_2026_th.md`, `evaluation_2026_manifest.json`.
- `data/v3/evaluation_2026/` (gitignored): `row_ledger.csv` (internal provenance ids, no coordinates), `cells_{daily,6h}.csv`, `predictions_{daily,6h}.csv` for the independent verifier.

Verify independently: `.venv/bin/python backend/scripts/verify_spatial_2026_evaluation.py --run-root backend/rerun-2026-eval-<id>`.
